# Breast Cancer Classification

## Цель проекта

Построить модель бинарной классификации для определения злокачественной
опухоли по числовым характеристикам обследования.

- Положительный класс (`1`) — malignant.
- Отрицательный класс (`0`) — benign.
- Главная ошибка — False Negative: модель относит злокачественную опухоль
  к доброкачественным.
- Основная метрика — Recall положительного класса.
- Дополнительные метрики — Precision, F1 и ROC-AUC.

Тестовая выборка ранее использовалась в учебных экспериментах, поэтому её результат не является полностью независимой оценкой окончательного выбора модели.

## 1. Импорты и настройки

In [24]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    GridSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    make_scorer
)
from sklearn.utils.class_weight import compute_sample_weight

RANDOM_STATE = 42



cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

scoring = {
    "precision": make_scorer(
        precision_score,
        zero_division=0
    ),
    "recall": make_scorer(recall_score),
    "f1": make_scorer(f1_score),
    "roc_auc": "roc_auc"
}

## 2. Загрузка и проверка данных

In [25]:
data = load_breast_cancer(as_frame=True)

X = data.data.copy()

# В исходном датасете:
# 0 — malignant, 1 — benign.
# Назначаем опасный класс положительным:
y = (data.target == 0).astype(int)
y.name = "malignant"

class_summary = pd.DataFrame({
    "count": y.value_counts().sort_index(),
    "share": y.value_counts(normalize=True).sort_index()
})
class_summary.index = ["benign (0)", "malignant (1)"]

print("Размер X:", X.shape)
print("Пропущенных значений:", X.isna().sum().sum())
print("Полных дубликатов:", X.duplicated().sum())
display(class_summary)

Размер X: (569, 30)
Пропущенных значений: 0
Полных дубликатов: 0


,count,share
benign (0),357,0.627417
malignant (1),212,0.372583


## 3. Разделение данных

In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)
print("Доля класса 1 в train:", y_train.mean())
print("Доля класса 1 в test:", y_test.mean())

Train: (455, 30)
Test: (114, 30)
Доля класса 1 в train: 0.37362637362637363
Доля класса 1 в test: 0.3684210526315789


## 4. Baseline и сравнение моделей

In [16]:
models = {
    "Dummy": DummyClassifier(
        strategy="most_frequent"
    ),

    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=3,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingClassifier(
    random_state=RANDOM_STATE
  )
}

cv_rows = []

for name, estimator in models.items():
    scores = cross_validate(
        estimator,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        n_jobs=-1
    )

    cv_rows.append({
        "model": name,
        "train_f1": scores["train_f1"].mean(),
        "cv_precision": scores["test_precision"].mean(),
        "cv_recall": scores["test_recall"].mean(),
        "cv_f1": scores["test_f1"].mean(),
        "cv_f1_std": scores["test_f1"].std(),
        "cv_roc_auc": scores["test_roc_auc"].mean()
    })

cv_comparison = (
    pd.DataFrame(cv_rows)
    .sort_values(["cv_recall", "cv_f1"], ascending=False)
    .reset_index(drop=True)
)

display(cv_comparison.round(3))

,model,train_f1,cv_precision,cv_recall,cv_f1,cv_f1_std,cv_roc_auc
0,Logistic Regression,0.984,0.965,0.959,0.962,0.024,0.995
1,Gradient Boosting,1.000,0.965,0.947,0.955,0.020,0.991
2,Random Forest,1.000,0.959,0.941,0.949,0.018,0.989
3,Decision Tree,0.966,0.893,0.912,0.900,0.038,0.918
4,Dummy,0.000,0.000,0.000,0.000,0.000,0.500


## 5. Подбор гиперпараметров

### 5.1 Logistic Regression

In [17]:
logreg_for_search = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE
    ))
])

logreg_param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__class_weight": [None, "balanced"]
}

logreg_search = GridSearchCV(
    estimator=logreg_for_search,
    param_grid=logreg_param_grid,
    scoring=scoring,
    refit="recall",
    cv=cv,
    return_train_score=True,
    n_jobs=-1
)

logreg_search.fit(X_train, y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('model',
                                        LogisticRegression(max_iter=2000,
                                                           random_state=42))]),
             n_jobs=-1,
             param_grid={'model__C': [0.01, 0.1, 1, 10, 100],
                         'model__class_weight': [None, 'balanced']},
             refit='recall', return_train_score=True,
             scoring={'f1': make_scorer(f1_score, response_method='predict'),
                      'precision': make_scorer(precision_score, response_method='predict', zero_division=0),
                      'recall': make_scorer(recall_score, response_method='predict'),
                      'roc_auc': 'roc_auc'})

In [18]:
logreg_results = pd.DataFrame(logreg_search.cv_results_)

logreg_columns = [
    "param_model__C",
    "param_model__class_weight",
    "mean_train_f1",
    "mean_test_precision",
    "mean_test_recall",
    "mean_test_f1",
    "std_test_f1",
    "mean_test_roc_auc"
]

display(
    logreg_results[logreg_columns]
    .sort_values(
        ["mean_test_recall", "mean_test_f1"],
        ascending=False
    )
    .round(3)
)

print("Автоматически выбраны:", logreg_search.best_params_)

,param_model__C,param_model__class_weight,mean_train_f1,mean_test_precision,mean_test_recall,mean_test_f1,std_test_f1,mean_test_roc_auc
5,1.00,balanced,0.984,0.965,0.959,0.962,0.024,0.995
3,0.10,balanced,0.978,0.983,0.953,0.967,0.011,0.995
4,1.00,None,0.984,0.977,0.953,0.964,0.021,0.996
7,10.00,balanced,0.987,0.960,0.947,0.953,0.018,0.994
8,100.00,None,0.999,0.949,0.947,0.947,0.015,0.991
9,100.00,balanced,0.999,0.949,0.947,0.947,0.015,0.992
6,10.00,None,0.987,0.970,0.941,0.955,0.017,0.994
2,0.10,None,0.975,0.989,0.935,0.960,0.022,0.995
1,0.01,balanced,0.963,0.970,0.935,0.952,0.023,0.992
0,0.01,None,0.935,0.994,0.865,0.923,0.029,0.991


Автоматически выбраны: {'model__C': 1, 'model__class_weight': 'balanced'}


##Подбор Gradient Boosting

In [19]:
boost_search = GridSearchCV(
    estimator=GradientBoostingClassifier(random_state=RANDOM_STATE),
    param_grid={
        "n_estimators": [100, 200],
        "learning_rate": [0.03, 0.1],
        "max_depth": [1, 2, 3],
        "min_samples_leaf": [3, 10],
    },
    scoring=scoring,
    refit="recall",
    cv=cv,
    return_train_score=True,
    n_jobs=-1,
)

boost_search.fit(X_train, y_train)

boost_results = pd.DataFrame(boost_search.cv_results_)

display(
    boost_results[[
        "param_n_estimators",
        "param_learning_rate",
        "param_max_depth",
        "param_min_samples_leaf",
        "mean_train_f1",
        "mean_test_precision",
        "mean_test_recall",
        "mean_test_f1",
        "std_test_f1",
        "mean_test_roc_auc",
    ]]
    .sort_values(["mean_test_recall", "mean_test_f1"], ascending=False)
    .head(10)
    .round(3)
)

print("Автоматически выбраны:", boost_search.best_params_)

,param_n_estimators,param_learning_rate,param_max_depth,param_min_samples_leaf,mean_train_f1,mean_test_precision,mean_test_recall,mean_test_f1,std_test_f1,mean_test_roc_auc
19,200,0.10,2,10,1.000,0.983,0.959,0.970,0.009,0.994
15,200,0.10,1,10,0.995,0.959,0.959,0.959,0.006,0.994
18,100,0.10,2,10,1.000,0.989,0.953,0.970,0.013,0.992
22,100,0.10,3,10,1.000,0.977,0.953,0.964,0.018,0.992
7,200,0.03,2,10,0.996,0.971,0.953,0.962,0.018,0.991
20,100,0.10,3,3,1.000,0.965,0.953,0.958,0.018,0.991
17,200,0.10,2,3,1.000,0.983,0.947,0.964,0.012,0.993
16,100,0.10,2,3,1.000,0.976,0.947,0.961,0.016,0.992
23,200,0.10,3,10,1.000,0.977,0.947,0.961,0.016,0.993
13,200,0.10,1,3,0.994,0.959,0.947,0.953,0.014,0.993


Автоматически выбраны: {'learning_rate': 0.1, 'max_depth': 1, 'min_samples_leaf': 10, 'n_estimators': 200}


In [20]:
print(
    boost_results.loc[
        [19, 15],
        [
            "mean_test_recall",
            "mean_test_precision",
            "mean_test_f1",
            "std_test_f1",
        ],
    ].to_string(float_format=lambda x: f"{x:.12f}")
)

    mean_test_recall  mean_test_precision   mean_test_f1    std_test_f1
19    0.958823529412       0.982689075630 0.970315708137 0.009043663069
15    0.958823529412       0.959139292080 0.958815894950 0.005914717800


In [21]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix

selected_boost = GradientBoostingClassifier(
    **boost_search.cv_results_["params"][19],
    random_state=RANDOM_STATE,
)

oof_proba = cross_val_predict(
    selected_boost,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1,
)[:, 1]

oof_pred = (oof_proba >= 0.5).astype(int)

print("Матрица ошибок (out-of-fold):")
print(confusion_matrix(y_train, oof_pred))

errors = X_train.copy()
errors["true_class"] = y_train
errors["predicted_class"] = oof_pred
errors["probability_malignant"] = oof_proba

fn = errors[
    (errors["true_class"] == 1) &
    (errors["predicted_class"] == 0)
]

fp = errors[
    (errors["true_class"] == 0) &
    (errors["predicted_class"] == 1)
]

print("FN:", len(fn), "| FP:", len(fp))
display(
    fn[["mean radius", "mean texture", "probability_malignant"]]
    .sort_values("probability_malignant", ascending=False)
)

Матрица ошибок (out-of-fold):
[[282   3]
 [  7 163]]
FN: 7 | FP: 3


,mean radius,mean texture,probability_malignant
146,11.80,16.58,0.353672
536,14.27,22.55,0.106497
41,10.95,21.35,0.044427
40,13.44,21.58,0.032285
255,13.96,17.05,0.019383
135,12.77,22.47,0.001987
297,11.76,18.14,0.000488


In [22]:
threshold_rows = []

for threshold in [0.1, 0.2, 0.3, 0.4, 0.5]:
    pred = (oof_proba >= threshold).astype(int)
    tn, fp_count, fn_count, tp = confusion_matrix(y_train, pred).ravel()

    threshold_rows.append({
        "threshold": threshold,
        "FP": fp_count,
        "FN": fn_count,
        "precision": precision_score(y_train, pred),
        "recall": recall_score(y_train, pred),
        "f1": f1_score(y_train, pred),
    })

display(pd.DataFrame(threshold_rows).round(3))

,threshold,FP,FN,precision,recall,f1
0,0.1,16,5,0.912,0.971,0.940
1,0.2,11,6,0.937,0.965,0.951
2,0.3,7,6,0.959,0.965,0.962
3,0.4,4,7,0.976,0.959,0.967
4,0.5,3,7,0.982,0.959,0.970


## Итоговая модель

In [23]:
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

final_model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=2,
    min_samples_leaf=10,
    random_state=RANDOM_STATE,
)

# Обучаем выбранную конфигурацию на всей обучающей выборке
final_model.fit(X_train, y_train)

test_proba = final_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test, test_pred))
print("Precision:", precision_score(y_test, test_pred))
print("Recall:", recall_score(y_test, test_pred))
print("F1:", f1_score(y_test, test_pred))
print("ROC-AUC:", roc_auc_score(y_test, test_proba))
print("Матрица ошибок:")
print(confusion_matrix(y_test, test_pred))

Accuracy: 0.9736842105263158
Precision: 1.0
Recall: 0.9285714285714286
F1: 0.9629629629629629
ROC-AUC: 0.9953703703703705
Матрица ошибок:
[[72  0]
 [ 3 39]]


## Выводы

В проекте сравнивались DummyClassifier, Logistic Regression,
Decision Tree, Random Forest и Gradient Boosting. Модели оценивались
с помощью стратифицированной 5-кратной кросс-валидации на обучающей
выборке. Основной метрикой был recall класса 1 - злокачественной опухоли.

Среди базовых моделей лучший recall показала Logistic Regression:
0.959. После подбора параметров Gradient Boosting достиг такого же
CV recall (0.959), но более высокого precision (0.983) и F1 (0.970).
По этим результатам для итогового обучения выбран Gradient Boosting
с 200 деревьями, learning_rate=0.1, max_depth=2 и
min_samples_leaf=10.

Анализ out-of-fold предсказаний при пороге 0.5 показал 7 FN и 3 FP.
Снижение порога до 0.3 уменьшило число FN до 6, но увеличило число
FP до 7. Поскольку допустимая цена этих ошибок не задана, для
итоговой модели оставлен порог 0.5.

Выбранная модель обучена на всей обучающей выборке. На тестовой
выборке из 114 объектов она получила precision=1.000,
recall=0.929, F1=0.963 и ROC-AUC=0.995. Из 42 злокачественных
случаев модель обнаружила 39 и пропустила 3. Матрица ошибок:
TN=72, FP=0, FN=3, TP=39.

### Ограничения

Датасет и особенно тестовая выборка невелики, поэтому результаты
могут заметно меняться на других данных. Кроме того, тестовая
выборка ранее уже использовалась в ходе учебного исследования:
её результат нельзя считать полностью независимой оценкой
окончательного выбора модели. Это учебный проект, а не система
для медицинской диагностики.

In [27]:
from pathlib import Path
import joblib
import sklearn

artifact = {
    "model": final_model,
    "feature_names": X_train.columns.tolist(),
    "class_mapping": {0: "benign", 1: "malignant"},
    "threshold": 0.5,
    "sklearn_version": sklearn.__version__,
}

model_path = Path("breast_cancer_model.joblib")
joblib.dump(artifact, model_path)

print("Модель сохранена:", model_path.resolve())

Модель сохранена: /content/breast_cancer_model.joblib


In [28]:
import numpy as np

loaded_artifact = joblib.load(model_path)

sample = X_train.iloc[:3][loaded_artifact["feature_names"]]

original_proba = final_model.predict_proba(sample)
loaded_proba = loaded_artifact["model"].predict_proba(sample)

np.testing.assert_allclose(original_proba, loaded_proba)

print("Проверка сохранённой модели: OK")
print("Классы:", loaded_artifact["class_mapping"])
print("Порог:", loaded_artifact["threshold"])

Проверка сохранённой модели: OK
Классы: {0: 'benign', 1: 'malignant'}
Порог: 0.5
